# Using Machine Learning techniques to predict a stroke

## Imports

In [478]:
import kagglehub
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report
from sklearn.base import clone
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.pipeline import Pipeline

## Load in data, basic data information

In [479]:
path = kagglehub.dataset_download("najmilfuad/oral-cancer-data-set")

file_path = os.path.join(path, "oral_cancer_prediction_dataset.csv")
df = pd.read_csv(file_path)

df.head()

,ID,Country,Age,Gender,Tobacco Use,Alcohol Consumption,HPV Infection,Betel Quid Use,Chronic Sun Exposure,Poor Oral Hygiene,...,Difficulty Swallowing,White or Red Patches in Mouth,Tumor Size (cm),Cancer Stage,Treatment Type,"Survival Rate (5-Year, %)",Cost of Treatment (USD),Economic Burden (Lost Workdays per Year),Early Diagnosis,Oral Cancer (Diagnosis)
0,1,Italy,36,Female,Yes,Yes,Yes,No,No,Yes,...,No,No,0.000000,0,No Treatment,100.000000,0.00,0,No,No
1,2,Japan,64,Male,Yes,Yes,Yes,No,Yes,Yes,...,No,No,1.782186,1,No Treatment,83.340103,77772.50,177,No,Yes
2,3,UK,37,Female,No,Yes,No,No,Yes,Yes,...,No,Yes,3.523895,2,Surgery,63.222871,101164.50,130,Yes,Yes
3,4,Sri Lanka,55,Male,Yes,Yes,No,Yes,No,Yes,...,No,No,0.000000,0,No Treatment,100.000000,0.00,0,Yes,No
4,5,South Africa,68,Male,No,No,No,No,No,Yes,...,No,No,2.834789,3,No Treatment,44.293199,45354.75,52,No,Yes


In [480]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 84922 entries, 0 to 84921
Data columns (total 25 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   ID                                        84922 non-null  int64  
 1   Country                                   84922 non-null  str    
 2   Age                                       84922 non-null  int64  
 3   Gender                                    84922 non-null  str    
 4   Tobacco Use                               84922 non-null  str    
 5   Alcohol Consumption                       84922 non-null  str    
 6   HPV Infection                             84922 non-null  str    
 7   Betel Quid Use                            84922 non-null  str    
 8   Chronic Sun Exposure                      84922 non-null  str    
 9   Poor Oral Hygiene                         84922 non-null  str    
 10  Diet (Fruits & Vegetables Intake)         849

## Data cleaning

In [481]:
df.columns

Index(['ID', 'Country', 'Age', 'Gender', 'Tobacco Use', 'Alcohol Consumption',
       'HPV Infection', 'Betel Quid Use', 'Chronic Sun Exposure',
       'Poor Oral Hygiene', 'Diet (Fruits & Vegetables Intake)',
       'Family History of Cancer', 'Compromised Immune System', 'Oral Lesions',
       'Unexplained Bleeding', 'Difficulty Swallowing',
       'White or Red Patches in Mouth', 'Tumor Size (cm)', 'Cancer Stage',
       'Treatment Type', 'Survival Rate (5-Year, %)',
       'Cost of Treatment (USD)', 'Economic Burden (Lost Workdays per Year)',
       'Early Diagnosis', 'Oral Cancer (Diagnosis)'],
      dtype='str')

### Dropping columns
In this stage we get rid of features that are synonymous with cancer (eg. Tumor Size, Cancer Stage), and ID column, since it carries no useful information.

In [482]:
df = df.drop(df.columns[17:-2], axis="columns")
df = df.drop(df.columns[0], axis="columns")
df.head()

,Country,Age,Gender,Tobacco Use,Alcohol Consumption,HPV Infection,Betel Quid Use,Chronic Sun Exposure,Poor Oral Hygiene,Diet (Fruits & Vegetables Intake),Family History of Cancer,Compromised Immune System,Oral Lesions,Unexplained Bleeding,Difficulty Swallowing,White or Red Patches in Mouth,Early Diagnosis,Oral Cancer (Diagnosis)
0,Italy,36,Female,Yes,Yes,Yes,No,No,Yes,Low,No,No,No,No,No,No,No,No
1,Japan,64,Male,Yes,Yes,Yes,No,Yes,Yes,High,No,No,No,Yes,No,No,No,Yes
2,UK,37,Female,No,Yes,No,No,Yes,Yes,Moderate,No,No,No,No,No,Yes,Yes,Yes
3,Sri Lanka,55,Male,Yes,Yes,No,Yes,No,Yes,Moderate,No,No,Yes,No,No,No,Yes,No
4,South Africa,68,Male,No,No,No,No,No,Yes,High,No,No,No,No,No,No,No,Yes


### Checking data for unique values

Before modifying the data to be fit for training we should check how many unique values each feature can have, so that we can determine how to represent it (eg. use 0 and 1 for yes-no columns or one-hot encoding for those with more posiible values).

In [483]:
for column in df.columns:
    print(f"{column} values: {df[column].unique()}")

Country values: <StringArray>
[       'Italy',        'Japan',           'UK',    'Sri Lanka',
 'South Africa',       'Taiwan',          'USA',      'Germany',
       'France',    'Australia',       'Brazil',     'Pakistan',
        'Kenya',       'Russia',      'Nigeria',        'Egypt',
        'India']
Length: 17, dtype: str
Age values: [ 36  64  37  55  68  70  41  53  62  50  65  34  56  59  43  63  44  71
  51  47  58  57  54  67  31  66  48  61  46  49  60  74  42  73  69  35
  52  39  40  45  28  38  33  75  78  72  76  29  80  32  26  77  30  79
  82  89  23  22  81  18  24  83  25  86  21  87  19  27  17  85  84  20
  88  15  93  92  94  90  96  16  91 101  98]
Gender values: <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Tobacco Use values: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Alcohol Consumption values: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
HPV Infection values: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Betel Quid Use values: <String

### Checking for NaN values

We check if there are any values missing from the dataset.
- `df.isna()` – returns a DataFrame of True/False values (True if the value in the original DataFrame is NaN, False otherwise)  
- `df.isna().any()` – for each column, checks if it contains at least one True value  
- `df.isna().any().any()` – returns True if there is **at least one NaN** in the original DataFrame, False otherwise

In [484]:
df.isna().any().any()

np.False_

### Modifying the data

In [485]:
# Binary to 1/0
df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})

for column in df.columns:
    if np.isin("Yes", df[column].unique()):
        df[column] = df[column].map({"Yes": 1, "No": 0})

# One-Hot Encoding
df = pd.get_dummies(df, columns=["Country", "Diet (Fruits & Vegetables Intake)"])

# Convert all bool columns from OHE to int
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

df.head()

,Age,Gender,Tobacco Use,Alcohol Consumption,HPV Infection,Betel Quid Use,Chronic Sun Exposure,Poor Oral Hygiene,Family History of Cancer,Compromised Immune System,...,Country_Pakistan,Country_Russia,Country_South Africa,Country_Sri Lanka,Country_Taiwan,Country_UK,Country_USA,Diet (Fruits & Vegetables Intake)_High,Diet (Fruits & Vegetables Intake)_Low,Diet (Fruits & Vegetables Intake)_Moderate
0,36,0,1,1,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
1,64,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,37,0,0,1,0,0,1,1,0,0,...,0,0,0,0,0,1,0,0,0,1
3,55,1,1,1,0,1,0,1,0,0,...,0,0,0,1,0,0,0,0,0,1
4,68,1,0,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,1,0,0


In [486]:
# Move label to the rightmost side of the dataframe
cols = [c for c in df.columns if c != "Oral Cancer (Diagnosis)"] + ["Oral Cancer (Diagnosis)"]
df = df[cols]

### Renaming the columns

In this step we standardize the naming convention of features in the dataframe.

In [487]:
def get_renamed_column(name):
    # Remove parentheses and their contents
    name = name[:name.index("(")] + name[name.index(")") + 1:] if "(" in name else name
    name = name.strip()
    name = name.lower()
    name = name.replace(" ", "_")
    name = name.replace("__", "_")

    return name


def get_rename_dict(column_names):
    rename_dict = {}
    for name in column_names:
        rename_dict[name] = get_renamed_column(name)

    return rename_dict

In [488]:
df = df.rename(columns=get_rename_dict(df.columns))
df.head()

,age,gender,tobacco_use,alcohol_consumption,hpv_infection,betel_quid_use,chronic_sun_exposure,poor_oral_hygiene,family_history_of_cancer,compromised_immune_system,...,country_russia,country_south_africa,country_sri_lanka,country_taiwan,country_uk,country_usa,diet_high,diet_low,diet_moderate,oral_cancer
0,36,0,1,1,1,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
1,64,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,1,0,0,1
2,37,0,0,1,0,0,1,1,0,0,...,0,0,0,0,1,0,0,0,1,1
3,55,1,1,1,0,1,0,1,0,0,...,0,0,1,0,0,0,0,0,1,0
4,68,1,0,0,0,0,0,1,0,0,...,0,1,0,0,0,0,1,0,0,1


### Check for duplicates

In [489]:
any(df.duplicated())

True

There are only unique records in the dataset already, we may proceed.

### Check for class imbalance

Ideally here should be a similar number of records in each class.

In [490]:
df[df.columns[-1]].sum() / df[df.columns[-1]].size

np.float64(0.4986811426956501)

Since stroke was confirmed only in 5% of patients we will use a few different techniques deal with the imbalance.

## Preparing the train and test sets

### Train-test split

In [491]:
print(df.columns)

Index(['age', 'gender', 'tobacco_use', 'alcohol_consumption', 'hpv_infection',
       'betel_quid_use', 'chronic_sun_exposure', 'poor_oral_hygiene',
       'family_history_of_cancer', 'compromised_immune_system', 'oral_lesions',
       'unexplained_bleeding', 'difficulty_swallowing',
       'white_or_red_patches_in_mouth', 'early_diagnosis', 'country_australia',
       'country_brazil', 'country_egypt', 'country_france', 'country_germany',
       'country_india', 'country_italy', 'country_japan', 'country_kenya',
       'country_nigeria', 'country_pakistan', 'country_russia',
       'country_south_africa', 'country_sri_lanka', 'country_taiwan',
       'country_uk', 'country_usa', 'diet_high', 'diet_low', 'diet_moderate',
       'oral_cancer'],
      dtype='str')


In [492]:
columns_to_drop = df.columns[15:32] # drop countries
df = df.drop(columns=columns_to_drop, axis="columns")
df.columns

Index(['age', 'gender', 'tobacco_use', 'alcohol_consumption', 'hpv_infection',
       'betel_quid_use', 'chronic_sun_exposure', 'poor_oral_hygiene',
       'family_history_of_cancer', 'compromised_immune_system', 'oral_lesions',
       'unexplained_bleeding', 'difficulty_swallowing',
       'white_or_red_patches_in_mouth', 'early_diagnosis', 'diet_high',
       'diet_low', 'diet_moderate', 'oral_cancer'],
      dtype='str')

In [493]:
X = df.drop(columns=[df.columns[-1]]) 
y = df[df.columns[-1]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Create models

### Random Forests

In [494]:
random_forest_models = {
    "RandomForest_100_5": RandomForestClassifier(
        n_estimators=100,
        max_depth=5
    ),
    "RandomForest_50_5": RandomForestClassifier(
        n_estimators=50,
        max_depth=5
    ),
    "RandomForest_50_10": RandomForestClassifier(
        n_estimators=50,
        max_depth=10
    ),
    "RandomForest_500_3": RandomForestClassifier(
        n_estimators=500,
        max_depth=3
    ),
    "RandomForest_200_10": RandomForestClassifier(
        n_estimators=200,
        max_depth=10
    ),
    "RandomForest_300_7": RandomForestClassifier(
        n_estimators=300,
        max_depth=7
    ),
}

### XGBoost

In [495]:
xgboost_models = {
    "XGBoost_100_5_10": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        scale_pos_weight=10
    ),
    "XGBoost_50_5_10": XGBClassifier(
        n_estimators=50,
        learning_rate=0.2,
        max_depth=5,
        scale_pos_weight=10
    ),
    "XGBoost_100_10_5": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=10,
        scale_pos_weight=5
    ),
    "XGBoost_150_7_10": XGBClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=7,
        scale_pos_weight=10
    ),
    "XGBoost_80_15_5": XGBClassifier(
        n_estimators=80,
        learning_rate=0.3,
        max_depth=15,
        scale_pos_weight=5
    ),
    "XGBoost_200_8_8": XGBClassifier(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=8,
        scale_pos_weight=8
    )
}

### Neural Networks

In [496]:
input_shape = X_train.shape[1]

neural_networks_models = {
    "NN_64_32_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_32_32_32_1": Sequential([
        Dense(32, activation="sigmoid", input_shape=(input_shape,)),
        Dense(32, activation="sigmoid"),
        Dense(32, activation="sigmoid"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_64_64_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(64, activation="tanh"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_128_64_64_1": Sequential([
        Dense(128, activation="relu", input_shape=(input_shape,)),
        Dense(64, activation="relu"),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_64_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(1, activation="sigmoid")
    ]),
    "NN_64_32_64_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(32, activation="tanh"),
        Dense(64, activation="tanh"),
        Dense(1, activation="sigmoid")
    ]),
}

c:\Users\Dell\anaconda\envs\tensorflow\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Train models

#### Cross-validation pipeline

In [497]:
def validate_models(models, X, y):
    results = {}

    for name, model in models.items():
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])

        scores = cross_validate(pipeline, X, y, cv=5, scoring=["precision", "recall", "f1"])
        results[name] = {
            "precision": scores["test_precision"].mean(),
            "recall": scores["test_recall"].mean(),
            "f1": scores["test_f1"].mean(),
        }

    return results

#### Training pipeline

In [498]:
def train_models(models, X, y):

    trained_models = {}

    for name, model in models.items():
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])

        trained_pipeline = pipeline.fit(X, y)
        trained_models[name] = trained_pipeline

    return trained_models


##### SMOTE + Random Forest

In [ ]:
results = validate_models(random_forest_models, X_train, y_train)
for name, result in results.items():
    print(f"{name}: {result}\n")

In [ ]:
trained_models = train_models(random_forest_models, X_train, y_train)

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"{name} Accuracy: {accuracy}")
    print(f"{name} Precision: {precision}")
    print(f"{name} Recall: {recall}")
    print(f"{name} F1: {f1}\n")

RandomForest_100_5 Accuracy: 0.5004415660877245
RandomForest_100_5 Precision: 0.49554367201426025
RandomForest_100_5 Recall: 0.39596439169139463
RandomForest_100_5 F1: 0.44019265026060567

RandomForest_50_5 Accuracy: 0.5016190756549896
RandomForest_50_5 Precision: 0.4973219068023567
RandomForest_50_5 Recall: 0.4408308605341246
RandomForest_50_5 F1: 0.4673755741521424

RandomForest_50_10 Accuracy: 0.5038563438327937
RandomForest_50_10 Precision: 0.4998792270531401
RandomForest_50_10 Recall: 0.4912759643916914
RandomForest_50_10 F1: 0.4955402574079617

RandomForest_500_3 Accuracy: 0.4956726523403003
RandomForest_500_3 Precision: 0.4889133511558421
RandomForest_500_3 Recall: 0.3690207715133531
RandomForest_500_3 F1: 0.4205898268398268

RandomForest_200_10 Accuracy: 0.5013246982631734
RandomForest_200_10 Precision: 0.49714937286202965
RandomForest_200_10 Recall: 0.4657566765578635
RandomForest_200_10 F1: 0.4809412918249786

RandomForest_300_7 Accuracy: 0.5027377097438916
RandomForest_300_7

##### SMOTE + XGBoost

In [ ]:
results = validate_models(xgboost_models, X_train, y_train)
for name, result in results.items():
    print(f"{name}: {result}\n")

XGBoost_100_5_10: {'precision': np.float64(0.4993521052904085), 'recall': np.float64(0.9996462611754564), 'f1': np.float64(0.6660120128811828)}

XGBoost_50_5_10: {'precision': np.float64(0.4993226393903217), 'recall': np.float64(0.999557835159411), 'f1': np.float64(0.6659661823064417)}

XGBoost_100_10_5: {'precision': np.float64(0.4993016542065728), 'recall': np.float64(0.9897123536588758), 'f1': np.float64(0.6637474992496534)}

XGBoost_150_7_10: {'precision': np.float64(0.4993149278919756), 'recall': np.float64(0.9990272399577315), 'f1': np.float64(0.6658414990088735)}

XGBoost_80_15_5: {'precision': np.float64(0.49994375256523166), 'recall': np.float64(0.7822195794691397), 'f1': np.float64(0.609965529595783)}

XGBoost_200_8_8: {'precision': np.float64(0.49939602432385166), 'recall': np.float64(0.9992925440761391), 'f1': np.float64(0.6659725288234755)}



In [ ]:
trained_models = train_models(xgboost_models, X_train, y_train)

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"{name} Accuracy: {accuracy}")
    print(f"{name} Precision: {precision}")
    print(f"{name} Recall: {recall}")
    print(f"{name} F1: {f1}\n")

XGBoost_100_5_10 Accuracy: 0.4959670297321166
XGBoost_100_5_10 Precision: 0.49599528857479386
XGBoost_100_5_10 Recall: 0.9996439169139466
XGBoost_100_5_10 F1: 0.6630190907301712

XGBoost_50_5_10 Accuracy: 0.4959081542537533
XGBoost_50_5_10 Precision: 0.4959656045703516
XGBoost_50_5_10 Recall: 0.9995252225519288
XGBoost_50_5_10 F1: 0.6629664619744922

XGBoost_100_10_5 Accuracy: 0.49643803355902266
XGBoost_100_10_5 Precision: 0.49620673304883833
XGBoost_100_10_5 Recall: 0.9937091988130564
XGBoost_100_10_5 F1: 0.6618966675890422

XGBoost_150_7_10 Accuracy: 0.4959081542537533
XGBoost_150_7_10 Precision: 0.49596465390279826
XGBoost_150_7_10 Recall: 0.9992878338278932
XGBoost_150_7_10 F1: 0.6629133858267716

XGBoost_80_15_5 Accuracy: 0.49749779216956136
XGBoost_80_15_5 Precision: 0.49594036020076765
XGBoost_80_15_5 Recall: 0.7975074183976261
XGBoost_80_15_5 F1: 0.6115687434578801

XGBoost_200_8_8 Accuracy: 0.4959670297321166
XGBoost_200_8_8 Precision: 0.4959943449575872
XGBoost_200_8_8 Recal

### Using class weight to deal with the imbalance

#### Training Neural Networks

In [ ]:
def train_neural_networks(models, X, y):
    results = {}

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    for name, model in models.items():
        model_copy = clone_model(model)
        model_copy.set_weights(model.get_weights()) 
        
        model_copy.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["precision", "recall"])
        model_copy.fit(X_scaled, y, epochs=50, batch_size=32, verbose=0, validation_split=0.2)

        results[name] = model_copy

    return results, scaler

In [ ]:
trained_models, scaler = train_neural_networks(neural_networks_models, X_train, y_train)
X_test_scaled = scaler.transform(X_test)

for name, model in trained_models.items():
    y_prob = model.predict(X_test_scaled).flatten()

    y_pred = (y_prob > 0.5).astype(int) 

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"{name} Accuracy: {accuracy}")
    print(f"{name} Precision: {precision}")
    print(f"{name} Recall: {recall}")
    print(f"{name} F1: {f1}\n")

KeyboardInterrupt: 